# Phase 3.2: Multi-Agent Traffic Signal Control (MARL)
**Objective:** Train decentralized traffic light agents to create "green waves" for approaching emergency vehicles.

This notebook uses Independent Q-Learning with Parameter Sharing. Every signaled intersection in Kigali acts as an independent agent, observing its local queues and the presence of ambulances. All agents push their experiences into a shared memory buffer and update a single, shared neural network. This allows the entire city grid to learn cooperatively and rapidly.

In [1]:
import sys
import random
import logging
import numpy as np
import traci
from pathlib import Path

# Add project root to path
sys.path.append(str(Path.cwd().parent))

from src.environment.manager import SimulationManager
from src.agents.traffic_marl import MultiAgentTrafficController

# Configure basic logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# Define paths
net_path = Path("../data/processed/kigali.net.xml")
route_path = Path("../data/processed/kigali_traffic.rou.xml")
marl_model_save_path = Path("../models/marl_traffic_v1.pt")

## 1. The MARL Environment Wrapper
We create a class to interface with SUMO's TraCI API specifically for traffic lights. It extracts the 3-dimensional state `[Current Phase, Max Queue Length, Ambulance Approaching Flag]` for every intersection and calculates the reward based on civilian delays and ambulance momentum.

In [2]:
class TrafficEnvironment:
    def __init__(self, sim_manager: SimulationManager):
        self.sim = sim_manager
        self.tls_ids = []
        self.target_ambulance_id = None

    def initialize_intersections(self):
        """Finds all traffic lights in the network."""
        self.tls_ids = traci.trafficlight.getIDList()
        logging.info(f"Initialized MARL Agents on {len(self.tls_ids)} intersections.")

    def inject_ghost_ambulance(self):
        """Randomly selects a civilian car and turns it into an emergency vehicle for training."""
        vehicles = traci.vehicle.getIDList()
        if vehicles and self.target_ambulance_id not in vehicles:
            self.target_ambulance_id = random.choice(vehicles)
            # Make it visually distinct (Red) and give it emergency speed multipliers
            traci.vehicle.setColor(self.target_ambulance_id, (255, 0, 0, 255))
            traci.vehicle.setSpeedFactor(self.target_ambulance_id, 2.0)

    def get_state(self, tls_id: str) -> np.ndarray:
        """Extracts the [Phase, Queue, Ambulance_Flag] state for a specific intersection."""
        # 1. Current Phase (Normalized roughly)
        current_phase = traci.trafficlight.getPhase(tls_id)
        
        # 2. Max Queue Length on incoming lanes
        lanes = traci.trafficlight.getControlledLanes(tls_id)
        max_queue = 0
        amb_approaching = 0.0

        for lane in lanes:
            queue = traci.lane.getLastStepHaltingNumber(lane)
            max_queue = max(max_queue, queue)
            
            # 3. Check if our Ghost Ambulance is on this lane
            vehicles_on_lane = traci.lane.getLastStepVehicleIDs(lane)
            if self.target_ambulance_id in vehicles_on_lane:
                amb_approaching = 1.0

        # Normalize state values for the neural network
        return np.array([current_phase / 10.0, max_queue / 50.0, amb_approaching], dtype=np.float32)

    def apply_action(self, tls_id: str, action: int):
        """Applies the agent's decision: 0 (Keep), 1 (Switch Phase)."""
        if action == 1:
            current_phase = traci.trafficlight.getPhase(tls_id)
            
            # Fetch the total number of phases for this specific intersection's program
            logic = traci.trafficlight.getCompleteRedYellowGreenDefinition(tls_id)[0]
            num_phases = len(logic.phases)
            
            # Use modulo to smoothly loop back to phase 0 if we reach the end of the cycle
            next_phase = (current_phase + 1) % num_phases
            traci.trafficlight.setPhase(tls_id, next_phase)

    def calculate_reward(self, tls_id: str) -> float:
        """
        Rewards:
        - Small penalty for civilian queues (-0.5 per car).
        - Massive penalty if ambulance is stopped at this light (-100).
        - Massive reward if ambulance successfully clears this light (+50).
        """
        lanes = traci.trafficlight.getControlledLanes(tls_id)
        total_queue = sum([traci.lane.getLastStepHaltingNumber(lane) for lane in lanes])
        
        reward = -0.5 * total_queue

        if self.target_ambulance_id:
            for lane in lanes:
                vehicles = traci.lane.getLastStepVehicleIDs(lane)
                if self.target_ambulance_id in vehicles:
                    speed = traci.vehicle.getSpeed(self.target_ambulance_id)
                    if speed < 1.0: # Ambulance is stuck at a red light!
                        reward -= 100.0
                    elif speed > 5.0: # Ambulance is moving freely through the intersection!
                        reward += 50.0

        return reward

## 2. The Shared Training Loop
We step through the simulation. Every 5 simulation seconds, the traffic lights observe their environment, make a decision using the shared network, and apply it. They then store the transition in the shared memory buffer to learn.

In [3]:
# Hyperparameters
EPISODES = 50
SIMULATION_STEPS = 1000 # Length of each episode
DECISION_INTERVAL = 5   # Agents make a decision every 5 simulation seconds
BATCH_SIZE = 128
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.99

# Initialize Manager and Agent
sim_manager = SimulationManager(net_path=net_path, route_path=route_path, use_gui=False)
env = TrafficEnvironment(sim_manager)

# State: Phase, Queue, Amb_Flag (3 dims). Action: Keep or Switch (2 dims).
agent = MultiAgentTrafficController(state_dim=3, action_dim=2, lr=1e-3)
agent.load_model(marl_model_save_path)

epsilon = EPSILON_START
best_global_reward = -float('inf')

print("\n--- Starting MARL Traffic Training Loop ---")

for episode in range(1, EPISODES + 1):
    try:
        sim_manager.start()
        env.initialize_intersections()
        
        episode_reward = 0
        
        for step in range(SIMULATION_STEPS):
            sim_manager.step()
            env.inject_ghost_ambulance()
            
            # Agents only make decisions every few seconds to allow phases to run
            if step % DECISION_INTERVAL == 0:
                states = {}
                actions = {}
                
                # 1. Observe and Act
                for tls_id in env.tls_ids:
                    state = env.get_state(tls_id)
                    action = agent.select_action(state, epsilon)
                    env.apply_action(tls_id, action)
                    
                    states[tls_id] = state
                    actions[tls_id] = action
                    
                # We must step the simulation 1 more time to see the *result* of the actions
                sim_manager.step()
                
                # 2. Calculate Reward and Store Experience
                for tls_id in env.tls_ids:
                    next_state = env.get_state(tls_id)
                    reward = env.calculate_reward(tls_id)
                    done = step >= SIMULATION_STEPS - DECISION_INTERVAL
                    
                    episode_reward += reward
                    agent.push_experience(states[tls_id], actions[tls_id], reward, next_state, done)
                    
                # 3. Train the Shared Network
                agent.update(BATCH_SIZE)
                
        # Sync target network
        if episode % 5 == 0:
            agent.update_target_network()
            
        epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
        
        # Save best model
        if episode_reward > best_global_reward:
            best_global_reward = episode_reward
            agent.save_model(marl_model_save_path)
            
        print(f"Episode {episode}/{EPISODES} | Shared Network Reward: {episode_reward:.2f} | Epsilon: {epsilon:.3f}")
        
    except Exception as e:
        print(f"Episode {episode} crashed: {e}")
    finally:
        sim_manager.close()

print("--- MARL Training Complete ---")

2026-03-24 11:36:51,681 - INFO - MARL Traffic Controller initialized on device: mps
2026-03-24 11:36:52,293 - INFO - Starting SUMO Simulation Engine...



--- Starting MARL Traffic Training Loop ---
 Retrying in 1 seconds


pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
2026-03-24 11:36:56,311 - INFO - Initialized MARL Agents on 128 intersections.
/var/folders/p7/mhf8qhy5037btnt6w33q98_r0000gn/T/ipykernel_19148/1129383093.py:49: UserWarning: Call to deprecated function getAllProgramLogics, use getCompleteRedYellowGreenDefinition instead.
  logic = traci.trafficlight.getCompleteRedYellowGreenDefinition(tls_id)[0]
2026-03-24 11:38:57,722 - INFO - Traffic MARL model saved to ../models/marl_traffic_v1.pt


Episode 1/50 | Shared Network Reward: -2435.50 | Epsilon: 0.990


2026-03-24 11:38:58,229 - INFO - SUMO simulation closed cleanly.
2026-03-24 11:38:58,229 - INFO - Starting SUMO Simulation Engine...


...

 Retrying in 1 seconds


pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
pj_obj_create: Cannot find proj.db
2026-03-24 13:26:16,749 - INFO - Initialized MARL Agents on 128 intersections.
/var/folders/p7/mhf8qhy5037btnt6w33q98_r0000gn/T/ipykernel_19148/1129383093.py:49: UserWarning: Call to deprecated function getAllProgramLogics, use getCompleteRedYellowGreenDefinition instead.
  logic = traci.trafficlight.getCompleteRedYellowGreenDefinition(tls_id)[0]


Episode 50/50 | Shared Network Reward: -23.00 | Epsilon: 0.605


2026-03-24 13:28:42,707 - INFO - SUMO simulation closed cleanly.


--- MARL Training Complete ---
